# Classification of Astronomical Events using CNNs (III)
```Authors: Paul Alvarez updated: 20260730```

This study implements and evaluates various computer vision models to classify astronomical events from the Zwicky Transient Facility (ZTF) survey. The main objective is to perform real-bogus classification using 63×63 pixel triplet images (science, reference, and difference) and analyze its performance in a reduced sample size. Additionally, it extends to multiclass classification (transient, periodic, stochastic) and includes transfer learning experiments using MobileNetV2 and Braai.


This third notebook covers the construction of the dataset for the multi class study, repeating the process seeing in notebook I and testing our best CNN of the previous notebook on the neural network.

## Table of contents:
* [Required libraries](#Required-libraries)
* [Dataset construction](#Dataset)
* [Download stamps](#initial-naive-approach)
* [Cleaning and Normalization](#cleaning-and-normalization)
* [Class separation](#class-separation)
* [References](#References)

## Required libraries

In [2]:
from alerce.core import Alerce
from astropy.io import fits
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
from functions import download_stamp, visual_fits, clean_labels, normalize_img, build_Xy
from astropy.io.fits.verify import VerifyWarning
from sklearn.preprocessing import LabelEncoder

import warnings
import json
import os
import time
import sqlalchemy as sa
import requests
import pandas as pd
import numpy as np

warnings.simplefilter('ignore', category=VerifyWarning)

## Dataset construction

In [5]:
# Connect to the ALeRCE API according to their instructions
client = Alerce()

url = "https://raw.githubusercontent.com/alercebroker/usecases/master/alercereaduser_v4.json"
params = requests.get(url).json()['params']

engine = sa.create_engine(f"postgresql+psycopg2://{params['user']}:{params['password']}@{params['host']}/{params['dbname']}")
engine.begin()

In [6]:
# API queries to obtain data
query_multiclass = """
(
    SELECT DISTINCT ON (p.oid)
           p.oid,
           p.class_name,
           p.probability
    FROM probability p
    WHERE p.classifier_name = 'lc_classifier_top'
      AND p.class_name = 'Transient'
      AND p.probability >= 0.80
    ORDER BY p.oid, p.probability DESC
    LIMIT 2000
)
UNION ALL
(
    SELECT DISTINCT ON (p.oid)
           p.oid,
           p.class_name,
           p.probability
    FROM probability p
    WHERE p.classifier_name = 'lc_classifier_top'
      AND p.class_name = 'Stochastic'
      AND p.probability >= 0.80
    ORDER BY p.oid, p.probability DESC
    LIMIT 2000
)
UNION ALL
(
    SELECT DISTINCT ON (p.oid)
           p.oid,
           p.class_name,
           p.probability
    FROM probability p
    WHERE p.classifier_name = 'lc_classifier_top'
      AND p.class_name = 'Periodic'
      AND p.probability >= 0.80
    ORDER BY p.oid, p.probability DESC
    LIMIT 2000
);
"""


df_mc = pd.read_sql_query(query_multiclass, engine)
df_mc["class_name"].value_counts(normalize=True) * 100

class_name
Transient     33.333333
Stochastic    33.333333
Periodic      33.333333
Name: proportion, dtype: float64

In [ ]:
# merge the queries and set labels to different classes
df_mc = df_mc.sample(frac=1, random_state=42).reset_index(drop=True)
df_mc['label'] = LabelEncoder().fit_transform(df_mc['class_name']) # 0, 1, 2

df_mc[['oid','label']].to_csv("labels_multiclase_01.csv", index=False)
df_mc.head()